<h1 align="center">GPR Testing</h1>

## **Pip Installs**

pip install matplotlib (3.10.7)

pip install numpy (2.3.5)

pip install scikit_learn(1.7.2)

pip install scipy(1.16.3)

pip install pandas(2.2.3)

In [306]:
# Install the requirements
%pip install matplotlib numpy scikit_learn scipy pandas

Note: you may need to restart the kernel to use updated packages.


## About GPR Algorithm

Think of GPR as a "smart" interpolation method that uses Bayesian probability.
 Instead of fitting a rigid function like a polynomial, you define a kernel, which acts as a prior describing the expected properties of the function (e.g., "I expect this curve to be very smooth").
  The GPR then calculates the most probable function (the "posterior") that fits your observed data points, given those properties. The key benefit is that it doesn't just return a single "best-fit" line;
   it returns a full probability distribution, giving you the mean prediction and a confidence interval (variance) for how certain it is.

## Content of Notebook:

1) Function that Creates a signal(radio astronomy observation)

2) Function that Estimate the Hyperparameters that will be used as initial values for the GPR

3) Function that create a mask at the spectral line area

4) Function that implement the GPR algorithm

5) Background Removal to signals and error estimation

## Import Libraries

In [307]:
import time
from enum import Enum

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import binary_dilation, gaussian_filter1d, label
from scipy.signal import savgol_filter

# We need GPR and Kernels from scikit-learn
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Create Signal(Background + Spectral line + Noise)

In [308]:

class SignalType(Enum):
    POLY=1
    SIN=2
    BOTH=3

def background_creation(freq : np.ndarray ) -> tuple :
    """
    Create a signal with bacground,spectral line and noise.The whole procedure will be random.

    Args:
        freq (np.ndarray): the frequencies of the signal

    Returns:
        tuple: the spectral line and the full observation
    """
    rng = np.random.default_rng()

    # --- 1)Background Signal Creation ---
    possible_signals = [SignalType.POLY,SignalType.SIN,SignalType.BOTH]
    signal_type = rng.choice(possible_signals)
    background = np.zeros(freq.shape)
    #  --- Creation of Baseline ---
    # is_polyonimal = True -> signal = a(frequencies - (freq1,freqn))^2 + b
    # is_sin = True -> signal = A* sin (ω  * (frequencies - (freq1,freqn)) + phase)
    # is_both = True -> signal = polyonimal + sin

    # Add the sin component if the signal is sin or has both sin and polyonimal compoonents
    if signal_type in (SignalType.SIN, SignalType.BOTH) :
        amplitude = rng.uniform(0.5 , 3)
        omega = 2 * np.pi * rng.uniform(low = 0.1 , high = 0.9)
        phase = rng.uniform(low = 0 , high = 2*np.pi)
        background += amplitude * np.sin(omega*(freq- (rng.uniform(np.min(freq) , np.max(freq))) ) + phase)

    #Add the poly component if the signal is poly or has both poly and sin components
    if signal_type in (SignalType.POLY , SignalType.BOTH):
        a = rng.uniform(low=-2 , high = 3)
        b = rng.uniform(low=3 , high = 7)
        background+= a * (freq-(rng.uniform(np.min(freq) , np.max(freq))))**2 + b

    # --- 2) Noise Creation ---
    noise = rng.normal(0.1 , 0.7 , freq.shape)

    # ---3) Spectral Line Creation ---
    # The spectral line will have a random mean between 1419.5 - 1421 MHz . random Amplitude between 2-8 and random standar deviation between 0.1 - 0.3
    # Spectra line formula: A * e^((x-m)^2 / 2sigma^2)
    mean = rng.uniform(1419.5 , 1421)
    sigma = rng.uniform(0.05 , 0.1)
    amplitude = rng.uniform(3 , 10)
    spectral_line = amplitude * np.exp(- ((freq-mean)**2) / (2 * sigma ** 2))

    return background,spectral_line,background + noise + spectral_line


## Estimate HyperParameters

In [309]:
def estimate_hyperparameters(signal: np.ndarray, dx : float) -> tuple :
    """
    Estimate The hyper parameters as close as we can so the GPR can have a good initial estimation.

    Args:
        signal (np.ndarray): The signal
        dx (float): the sampling step

    Returns:
        tuple: the hyper parameters
    """
     # --- Noise std Approach ---
    noise_guess = np.median(np.abs(np.diff(signal)))

    # --- Signal Variance Approach ---
    total_var = np.var(signal)
    signal_var = total_var - noise_guess**2

    # --- Length Scale Approach ---
    # Create the gaussian filter
    linewidth_bins = signal_var
    smoothing_linewidth = 5.0*linewidth_bins
    signal_smooth = gaussian_filter1d(signal,sigma=smoothing_linewidth)

    # center the signal so the autocorrelation is around 0
    y_centered = signal_smooth - np.mean(signal_smooth)
    corr = np.correlate(y_centered,y_centered,mode="full")

    # The result of the mode = full  is symetric.The point in the middle (size//2) is the Lag 0.We are intersted in the distance as we move forward(Lag >=0).
    corr = corr[corr.size//2:]

    # Normalize the data of the correlation with the lag 0 (corr[0]) because at lag 0 the signal completely identifies with itself and the value is the variance of the signal.
    # We want a coefficient from 0 to 1 therefore doing corr/=corr[0] we force the curve to start from 1
    corr /=corr[0]

    # we save the indexes that has correlation < 0.368
    one_e = 0.368
    below = np.where(corr<one_e)[0]


    if len(below)>0 :
        # We will define the interpolation boundaries

        idx2 = below[0] # The first point under the 0.368
        idx1 = idx2 - 1 # The last point over 0.368

        if idx1 < 0:
            #If the correlation was under 0.368 at the first step then the signal is almost just noise so we return the minimum possible length
            return noise_guess,signal_var,dx

        # Calculate the sub-pixel accuracy
        # The curve is a line between two points and we find where 0.368 is located in this line
        fraction = (corr[idx1]- 0.368/corr[idx1] - corr[idx2])

        #isx1 + fraction is the position of the length_scale in pixels
        return noise_guess , signal_var , (idx1+fraction)*dx

    # if the correlation was never below 0.368
    return noise_guess , signal_var , np.nan

## Implementation of the spectral line mask

### Method : **Rolling Median + MAD**

This method contains 3 steps:

Implementation Steps:

Calculate a Rolling Median to estimate the background while ignoring the peaks.

Calculate the MAD (Median Absolute Deviation) to estimate the noise level 
robustly.

Create a boolean mask where $|Data - Median| > Threshold \times \sigma_{MAD}$.

### Parameter Tuning

The main issue is the choice of the window size.The window must be wider than the spectral line (to avoid fitting the line) but narrower than the background curvature (to follow the baseline).

The choice of 1MHz as the window size is explained:

The bandwidth of the signal is 4MHz and if set as a target the Hydrogen line(HI) the simulated width of the line is approximantly 0.6 MHz.So the number 1 is enough to cover the spectral line and it also provides a safety margin

Using a threshold of 1 adopts an aggressive strategy: 'Flag any potential signal candidates (anything above 1σ) and rely on the size filter to discard isolated noise artifacts.'"

### Essential Mask Refinements for Optimal GPR Performance

Problem Definition: **The Edge Contamination Effect**. Once the initial mask is generated, a critical challenge arises regarding the Gaussian Process Regressor's (GPR) behavior within the masked gap. The GPR relies on the data immediately surrounding the gap to interpolate the missing values.

If the mask is not sufficiently wide, the GPR detects the "wings" (the rising and falling edges) of the spectral line at the boundaries of the mask. Consequently, the algorithm infers that an upward, convex curve exists within the gap, leading to an overestimation of the background power (an artificial "bump"). To prevent this, the mask must be widened (dilated) to completely obscure the spectral line's profile from the algorithm.

Implementing a blind expansion of the mask introduces a secondary issue. To ensure we capture the base of the spectral line, we utilize a deliberately low detection threshold (e.g., $1\sigma$). While effective for the signal, this aggressive threshold inevitably flags numerous isolated noise spikes across the spectrum as "masked" data. Expanding these random noise artifacts would fragment the dataset with unnecessary gaps, destabilizing the GPR.

The Solution: **Size Filtering (Morphological Cleaning)** To resolve the conflict between the need for a wide mask and the preservation of the noise floor, we implemented a Size Filtering mechanism (Connected Component Analysis):

Identification: The algorithm identifies continuous clusters of masked points.

Filtration: Clusters containing a small number of consecutive points are classified as random noise and are unmasked. Only large, continuous regions (characteristic of the spectral line) are retained.

Safe Dilation: With the noise artifacts removed, the mask now isolates only the spectral line.

This approach achieves two vital goals:

It leaves the general noise floor intact, allowing the GPR to accurately utilize the pre-calculated noise_level hyperparameter without statistical distortion.

It permits us to aggressively expand (pad) the mask around the spectral line without the risk of expanding noise artifacts, effectively solving the edge contamination problem.

## Rolling Median and MAD

In [310]:
def spectral_mask(y_observed: np.ndarray , freq : np.ndarray ) -> np.ndarray :
    """
    Create a mask to hide the spectral line data from the GPR Algorithm,implement rolling median and MAD filter.

    Args:
        y_observed (np.ndarray): The signal
        freq (np.ndarray): The frequencies axes

    Returns:
        np.ndarray: The mask
    """
    # Define the window and threshold according to the explanation above
    window_width_freq = 1 # MHz
    threshold_sigma = 1

    # Step 1: Convert the window width from MHz to bins(points) by finding the resolution
    freq_step = np.abs(freq[1] - freq[0])
    window_bins = int(window_width_freq / freq_step)

    # The window must be odd number
    if window_bins % 2 == 0:
        window_bins += 1

    # 2. Estimate Rolling Median
    y_series = pd.Series(y_observed)
    rolling_median = y_series.rolling(window=window_bins, center=True, min_periods=1).median().to_numpy()

    # 3. Estimate the Residuals
    diff = np.abs(y_observed - rolling_median)

    # 4. Estimation MAD (Median Absolute Deviation) and Sigma
    #  MAD show how much noise the signal has
    mad = np.median(diff)

    # Convert MAD to standar deviation (sigma) for normal distribution
    sigma = 1.4826 * mad

    # 5. Create Mask

    #Use the filter to all the signal
    """
    mask = diff > (threshold_sigma * sigma)

    """
     #Use the filter only between 1419 , 1421.3
    """
    mask = np.array(np.zeros_like(y_observed,bool),dtype=bool)
    for i in range(len(mask)):
        if(freq[i] >=1419 and freq[i]<=1421.3) and diff[i]>threshold_sigma*sigma:  # noqa: PLR2004
            mask[i]=True
    """

    # The points where the differnecies bigger from the threshold we set it true(masked)
    mask = diff>(threshold_sigma*sigma)
    # Remove the mask from the start and end of the signal
    mask[:50]= False
    mask[-50:] = False


    # 6. Remove the use of the filter at small areas which is noise
    # Method: (Size Filtering)

    # Create a unique label for each unique feature
    labeled_array , _num_features = label(mask)

    # Count number of occurrences of each value
    component_sizes = np.bincount(labeled_array.ravel())

    # Check which IDs have small duration,smaller than 15 points
    small_threshold = 15
    small_duration_components = component_sizes < small_threshold

    # Create the array that will show us which positions in our initial mask are noise and not the line area
    small_components_mask = small_duration_components[labeled_array]

    # Clean the mask
    clean_mask = mask.copy()
    clean_mask[small_components_mask] = False

    # Mask the beggining and the end of the spectral line so the GPR fills the gap with the background behaviour
    padding = 30

    return binary_dilation(clean_mask,iterations=padding)



In [311]:
time_res = []
res = []
for _i in range(100):
    # Frequencies axes
    frequencies = np.linspace(1418,1422,1024)
    # Create the signal and kepp the background and specral line for evaluation purposes
    background , spectral_line , y_observed = background_creation(freq=frequencies)

    start_time = time.time()
    mask = spectral_mask(y_observed=y_observed,freq=frequencies)

    # Mask the spectral line data
    X_train = frequencies[~mask].reshape(-1,1)
    y_train = y_observed[~mask]


    noise_guess , signal_var , length_scale_est = estimate_hyperparameters(y_observed,frequencies[1]-frequencies[0])


    kernel_foreground = signal_var*RBF(length_scale=length_scale_est,length_scale_bounds=(length_scale_est/4,length_scale_est*50))
    noise_kernel = WhiteKernel(noise_level=noise_guess,noise_level_bounds=(1e-5,noise_guess*1.3))
    total_kernel = kernel_foreground+noise_kernel
    gpr = GaussianProcessRegressor(kernel=total_kernel,alpha=0,optimizer="fmin_l_bfgs_b",n_restarts_optimizer=3)
    gpr.fit(X_train,y_train)
    baseline_gpr = gpr.predict(frequencies.reshape(-1,1))
    spectral_estimated = y_observed - baseline_gpr
    time_res.append(time.time() - start_time)

    # --- STEP 1: Calculate the Final Metrics ---
    spectral_estimated_smooth = savgol_filter(spectral_estimated,window_length=21,polyorder=1)
    rmse_line = np.sqrt(mean_squared_error(spectral_line, spectral_estimated_smooth))
    r2_line = r2_score(spectral_line, spectral_estimated_smooth)
    mae_line = mean_absolute_error(spectral_line, spectral_estimated_smooth)

    res.append((rmse_line,r2_line,mae_line))

    """
    # --- STEP 4: Generate Plots ---
    line_residuals = spectral_line - spectral_estimated_smooth
    # --- Plot 1: Direct Overlap ---
    plt.figure(figsize=(12, 6))
    plt.title("Plot 1: Y Observed")
    plt.plot(frequencies, y_observed, "b-", label="True Line (Ground Truth)", linewidth=2)
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Flux (Line only)")
    plt.legend()
    plt.grid(visible=True)
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.title("Plot 2: Signal with spectral mask")
    plt.plot(X_train, y_train, "b-", label="True Line (Ground Truth)", linewidth=2)
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Flux (Line only)")
    plt.legend()
    plt.grid(visible=True)
    plt.show()


    plt.figure(figsize=(12, 6))
    plt.title("Plot 3: Background vs. GPR_Background")
    plt.plot(frequencies, background, "b-", label="True Line (Ground Truth)", linewidth=2)
    plt.plot(frequencies, baseline_gpr, "r--", label="Extracted Line (from GPR)", linewidth=2)
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Flux (Line only)")
    plt.legend()
    plt.grid(visible=True)

    plt.show()

    plt.figure(figsize=(12, 6))
    plt.title("Plot 4: spectral line vs spectral line estimated")
    plt.plot(frequencies, spectral_line, "b-", label="True Line (Ground Truth)", linewidth=2)
    plt.plot(frequencies, spectral_estimated_smooth, "r--", label="Extracted Line (from GPR)", linewidth=2)
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Flux (Line only)")
    plt.legend()
    plt.grid(visible=True)
    plt.show()

    # --- Plot 2: Line Residuals ---
    plt.figure(figsize=(12, 6))
    plt.title("Plot 5: Final Extraction Error (Line Residuals)")
    plt.plot(frequencies, line_residuals, "o", markersize=2, label="Residual Error")
    plt.axhline(0, color="red", linestyle="--", label="Zero Error")
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Error (True - Extracted)")
    plt.legend()
    plt.grid(visible=True)
    plt.show()
    """


time_res = np.array(time_res)
print(f"Mean Duration = {np.mean(time_res)}")
rmse_sum = 0
r2_sum = 0
mse_sum = 0
for a,b,c in res:
    rmse_sum+=a
    r2_sum += b
    mse_sum += c

print(f"The mean ROOT MEAN SQUARED ERROR is: {rmse_sum/100 :.4f}")
print(f"The mean R2 Score is: {r2_sum/100 :.4f}")
print(f"The mean MEAN SQUARED ERROR is: {mse_sum/100 :.4f}")

KeyboardInterrupt: 